# 22 — E18 analysis: weight vs substitutability (zero solves; runs after 21)

Analyses **every E18 arm found in `runs/e18_*`** against its base scenario, S0 and the 12-position core.
**Pre-registered verdict rule per arm (frozen before the first run):**
- **SUBSTITUTABLE** if, in the arm's frequent tier, refugia's enrichment still exceeds the led block's (mean of its
  members) AND the tier's OWN land (outside the Act-1 core and the other scenarios' tiers) is < 3,000 km².
- **WEIGHT-LIMITED** if the led block's enrichment exceeds refugia's OR own land > 10,000 km².
- In between: AMBIGUOUS — report both numbers, no claim.
Closes with a dose table (influence share → tier km², own land, enrichments, max f, Gate-2b D, and the R10.14 lead-magnitude
currency = per-cell shortfall cost of the led block's 10,000 densest unprotected cells ÷ refugia's) → results_log R10.13/R10.14.
The fifth arm (`e18_s4x1_wonly`: S4's weight vector at S0's targets) is carbon's weights-only counterfactual (M4.22). Kernel `y2y-geo`.


**Pinned to VERSION v1 (study plan v0.17):** the E18 arms and their bases were solved on the 40-class block; they stand as evidence with disclosure and are never re-solved, so this notebook reads `runs/` and `spec/manifest.csv` regardless of `config.Y2Y_VERSION`.

In [1]:
import importlib, json, pathlib, sys
import numpy as np, pandas as pd, rasterio
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
ROOT = _cands[0]; sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec, director_core as dc
for _m in (config, lc, ec, dc): importlib.reload(_m)
G = dc.grid(); RUNS = config.y2y_paths("v1").runs; THR = dc.FREQ_THR     # E18 = v1 evidence (study plan v0.17)
SELFTEST = False          # True: analyse S2 against itself (exercises the code; identity expected)
MAN = dc.package_manifest(pd.read_csv(config.y2y_paths("v1").manifest))
VALS = {f: np.nan_to_num(lc._read(config.HANDOFF_DIR / f"{f}.tif")[G.pu], nan=0.0) for f in lc.continuous_features()}
CH = pd.read_csv(ROOT / "analyses/y2y/audit/audit_objects/feature_characterization.csv").set_index("feature")
def fsurf(d):
    A = ec.read_selections(d / "anchor.tif", G.pu)[0]
    S = np.vstack([A[None, :], ec.read_selections(d / "mga_guard_g05.tif", G.pu)])
    return A, S.mean(0), S
def diversity(S):
    # Gate-2b D: max PAIRWISE Hamming over anchor+members / (2 x discretionary cells selected) -- 1.0 = complete turnover
    X = S[:, G.disc].astype(np.float32); n = X.sum(1); H = n[:, None] + n[None, :] - 2 * (X @ X.T)
    return float(H.max() / (2 * (S[0] & G.disc).sum()))
TOT = {f: v.sum() for f, v in VALS.items()}
def lead_over_refugia(w, t, block, k=10_000):
    # R10.14 currency: per-cell min-shortfall cost of losing a cell, w_f (v_i/T_f) / t_f, mean over the block's k densest
    # unprotected cells, divided by refugia's under the same (w, t). Charged only while the feature sits BELOW target.
    cost = lambda feats: sum(w[f] * VALS[f] / TOT[f] / t.get(f, 1.0) for f in feats)
    top = lambda x: np.sort(x[G.disc])[::-1][:k].mean()
    return float(top(cost(config.BLOCKS[block])) / top(cost(config.BLOCKS["core_habitat"])))
def enrich(m):
    a = m.sum() / G.n_pu; return {f: float(v[m].sum() / v.sum()) / a for f, v in VALS.items()}
def capture(sel):
    return {f: float(v[sel].sum() / v.sum()) for f, v in VALS.items()}
def infl_share(w, t):
    inf = {f: w[f] * lc.swing_per_unit_w(CH.loc[f, "cap_min"], CH.loc[f, "cap_max"], t.get(f, 1.0)) for f in w}
    tot = sum(inf.values()); return {b: sum(inf[f] for f in fs) / tot for b, fs in config.BLOCKS.items()}
blk = lambda cap: {b: float(np.mean([cap[f] for f in fs])) for b, fs in config.BLOCKS.items()}
F_ALL = {fid: fsurf(RUNS / fid)[1] for fid in MAN.formulation_id}
core = (dc.ensemble(F_ALL, list(MAN.formulation_id)) >= THR) & G.disc
TIER = {sid: (np.mean([F_ALL[f] for f in F_ALL if f.startswith(sid + "_")], 0) >= THR) & G.disc for sid in ("s1", "s2", "s3", "s4")}
MANI = pd.read_csv(config.y2y_paths("v1").manifest).set_index("formulation_id")
A0, f0, _ = fsurf(RUNS / "s0_ssp585_theta5")
arms = sorted(RUNS.glob("e18_*")) if not SELFTEST else [RUNS / "s2_ssp585_theta5"]
print("arms found:", [a.name for a in arms])


arms found: ['e18_s2x2_ssp585', 'e18_s2x5_ssp585', 'e18_s3x2_ssp585', 'e18_s3x5_ssp585', 'e18_s4x1_wonly_ssp585']


In [2]:
# ---- per-arm verdicts (pre-registered rule) -----------------------------------------------------------
ROWS = []
for d in arms:
    meta = json.loads((d / "e18_meta.json").read_text()) if (d / "e18_meta.json").exists() else dict(base="s2_ssp585_theta5", block="connectivity", mult=1, weight_vector=json.loads(pd.read_csv(config.y2y_paths("v1").manifest).set_index("formulation_id").loc["s2_ssp585_theta5", "weight_vector"]), target_vector={})
    base, mult = meta["base"], meta["mult"]; block = meta.get("block", "connectivity")   # first run predates the field
    if not SELFTEST:
        cert = pd.read_csv(d / "certificates_guard.csv"); assert cert.band_ok.all() and len(cert) == 50, d.name
    Ab, fb, _ = fsurf(RUNS / base); Ax, fx, Sx = fsurf(d)
    tb, tx = (fb >= THR) & G.disc, (fx >= THR) & G.disc
    others = np.zeros(G.n_pu, bool)
    for sid, m in TIER.items():
        if sid != base[:2]: others |= m
    ownb, ownx = tb & ~core & ~others, tx & ~core & ~others
    eb, ex = enrich(tb), enrich(tx)
    lead_x = float(np.mean([ex[f] for f in config.BLOCKS[block]])); ref_x = ex["climate_type_macrorefugia"]
    share = infl_share(meta["weight_vector"], meta["target_vector"])[block]
    if lead_x > ref_x or ownx.sum() > 10_000:
        verdict = "WEIGHT-LIMITED"
    elif ref_x > lead_x and ownx.sum() < 3_000:
        verdict = "SUBSTITUTABLE"
    else:
        verdict = "AMBIGUOUS"
    capb, capx = blk(capture(Ab)), blk(capture(Ax))
    tf = meta.get("targets_from")
    print(f"\n===== {d.name}: {block} x{mult} on {base[:2].upper()}" + (f", targets from {tf[:2].upper()} (weights-only)" if tf else "") + f" (influence share {share:.2f}) =====")
    print(f"anchor {block} capture {capb[block]:.3f} -> {capx[block]:.3f} | core habitat {capb['core_habitat']:.3f} -> {capx['core_habitat']:.3f} | anchor Jaccard vs base {dc.jaccard(Ab & G.disc, Ax & G.disc):.3f}")
    print(f"frequent tier {tb.sum():,} -> {tx.sum():,} km2 | Jaccard {dc.jaccard(tb, tx):.3f} | inside 12-core {100*(tb&core).sum()/max(tb.sum(),1):.0f}% -> {100*(tx&core).sum()/max(tx.sum(),1):.0f}% | own land {ownb.sum():,} -> {ownx.sum():,} km2")
    print(f"tier enrichment: {block} {float(np.mean([eb[f] for f in config.BLOCKS[block]])):.2f} -> {lead_x:.2f} | refugia {eb['climate_type_macrorefugia']:.2f} -> {ref_x:.2f}")
    Dx = diversity(Sx); lead = lead_over_refugia(meta["weight_vector"], meta["target_vector"], block)
    print(f"max f (unprotected) {fx[G.disc].max():.3f} | D {Dx:.3f} | lead-magnitude {lead:.2f}x refugia")
    print(f"PRE-REGISTERED VERDICT: {verdict}")
    ROWS.append(dict(arm=d.name, block=block, mult=mult, targets_from=tf or "", influence_share=share, anchor_capture=capx[block], tier_km2=int(tx.sum()),
                     own_km2=int(ownx.sum()), enrich_led=lead_x, enrich_refugia=ref_x, max_f=float(fx[G.disc].max()), D=Dx,
                     lead_over_refugia=lead, verdict=verdict))
# base rows for the dose table
for sid, block in (("s2", "connectivity"), ("s3", "biodiversity"), ("s4", "carbon")):
    d = RUNS / f"{sid}_ssp585_theta{'3' if sid == 's4' else '5'}"
    row = MANI.loc[d.name]
    Ab, fb, Sb = fsurf(d); tb = (fb >= THR) & G.disc
    wb, tgb = json.loads(row.weight_vector), json.loads(row.target_vector)
    others = np.zeros(G.n_pu, bool)
    for s_, m in TIER.items():
        if s_ != sid: others |= m
    eb = enrich(tb)
    ROWS.append(dict(arm=d.name, block=block, mult=1, targets_from="", influence_share=infl_share(wb, tgb)[block],
                     anchor_capture=blk(capture(Ab))[block], tier_km2=int(tb.sum()), own_km2=int((tb & ~core & ~others).sum()),
                     enrich_led=float(np.mean([eb[f] for f in config.BLOCKS[block]])), enrich_refugia=eb["climate_type_macrorefugia"],
                     max_f=float(fb[G.disc].max()), D=diversity(Sb), lead_over_refugia=lead_over_refugia(wb, tgb, block), verdict="(base)"))
DOSE = pd.DataFrame(ROWS).sort_values(["block", "influence_share"]).reset_index(drop=True)
print("\nDOSE TABLE (→ results_log R10.13 / R10.14):")
print(DOSE.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
DOSE.to_csv(dc.SPEC / "E18_dose_table.csv", index=False)



===== e18_s2x2_ssp585: connectivity x2 on S2 (influence share 0.67) =====
anchor connectivity capture 0.335 -> 0.366 | core habitat 0.420 -> 0.383 | anchor Jaccard vs base 0.561
frequent tier 11,086 -> 12,687 km2 | Jaccard 0.245 | inside 12-core 81% -> 24% | own land 861 -> 7,925 km2
tier enrichment: connectivity 1.37 -> 2.30 | refugia 4.65 -> 2.44
PRE-REGISTERED VERDICT: AMBIGUOUS

===== e18_s2x5_ssp585: connectivity x5 on S2 (influence share 0.83) =====
anchor connectivity capture 0.335 -> 0.384 | core habitat 0.420 -> 0.340 | anchor Jaccard vs base 0.330
frequent tier 11,086 -> 34,705 km2 | Jaccard 0.054 | inside 12-core 81% -> 3% | own land 861 -> 31,780 km2
tier enrichment: connectivity 1.37 -> 2.12 | refugia 4.65 -> 0.88
PRE-REGISTERED VERDICT: WEIGHT-LIMITED

===== e18_s3x2_ssp585: biodiversity x2 on S3 (influence share 0.67) =====
anchor biodiversity capture 0.341 -> 0.349 | core habitat 0.407 -> 0.392 | anchor Jaccard vs base 0.708
frequent tier 6,471 -> 1,024 km2 | Jaccard 0

/var/folders/9r/qvrv12td3y9_rcssqr2csvbm0000gn/T/ipykernel_7864/4173092285.py:16: RuntimeWarning: invalid value encountered in scalar divide
  a = m.sum() / G.n_pu; return {f: float(v[m].sum() / v.sum()) / a for f, v in VALS.items()}



===== e18_s3x5_ssp585: biodiversity x5 on S3 (influence share 0.83) =====
anchor biodiversity capture 0.341 -> 0.363 | core habitat 0.407 -> 0.375 | anchor Jaccard vs base 0.429
frequent tier 6,471 -> 0 km2 | Jaccard 0.000 | inside 12-core 68% -> 0% | own land 521 -> 0 km2
tier enrichment: biodiversity 1.15 -> nan | refugia 4.24 -> nan
PRE-REGISTERED VERDICT: AMBIGUOUS

===== e18_s4x1_wonly_ssp585: carbon x1 on S4, targets from S0 (weights-only) (influence share 0.50) =====
anchor carbon capture 0.440 -> 0.370 | core habitat 0.418 -> 0.434 | anchor Jaccard vs base 0.595
frequent tier 34,787 -> 16,425 km2 | Jaccard 0.335 | inside 12-core 31% -> 72% | own land 20,329 -> 763 km2
tier enrichment: carbon 3.83 -> 1.40 | refugia 2.07 -> 4.03
PRE-REGISTERED VERDICT: SUBSTITUTABLE

DOSE TABLE (→ results_log R10.13):
                  arm        block  mult     targets_from  influence_share  anchor_capture  tier_km2  own_km2  enrich_led  enrich_refugia        verdict
     s3_ssp585_theta5 biodi